# AI应用基础 · 工单M3.2《产线实时缺陷检测系统》
**第9周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 尝试batch推理或输入裁剪优化，量化提速百分比并写结论；示范讲解 |
| **B 组长** | 标准达成 | FPS≥20且检出稳定；权衡分析有理 |
| **C 操作员** | 基础完成 | CPU推理+可视化完成；阈值实验完整 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：置信度阈值调高，漏检与误检各怎么变？你的机器CPU单帧约多少ms？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m3yolo"], check=True)
print("数据已生成: data/")

## 三、任务要求（Make）
**实施步骤**
1. 加载离线权重推理≥20张并可视化
2. conf∈{0.25,0.4,0.6,0.8}阈值实验
3. imgsz三档FPS测试(warmup后计时)
4. 填写精度-速度权衡矩阵

**验收标准**：及格(D)=`CPU推理+可视化完成；阈值实验完整` ；优秀(B+)=`FPS≥20且检出稳定；权衡分析有理`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
# -*- coding: utf-8 -*-
"""M3.2 产线实时缺陷检测（预训练YOLOv8n, CPU推理主线）—— 学生代码框架
自动适配两种环境: 装了ultralytics用YOLOv8n; 否则用onnxruntime+教师导出的ONNX。
"""
import glob
import time

import cv2
import numpy as np

IMGS = sorted(glob.glob("data/yolo_dataset/images/*.npy"))
CLASSES = ["dent", "stain", "edge_crack"]
W_PATH = "data/weights/yolov8n.pt"
ONNX = "data/weights/yolov8n.onnx"

### Starter 2：`def run_ultralytics(img_bgr, conf=0.4, imgsz=3`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def run_ultralytics(img_bgr, conf=0.4, imgsz=320):
    """方案A: ultralytics"""
    from ultralytics import YOLO
    model = YOLO(W_PATH)
    r = model.predict(img_bgr, conf=conf, imgsz=imgsz, device="cpu",
                      classes=None, verbose=False)
    return r[0].plot(), len(r[0].boxes)

### Starter 3：`def run_onnx(img_bgr, conf=0.4, imgsz=320):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def run_onnx(img_bgr, conf=0.4, imgsz=320):
    """方案B(离线备选): onnxruntime 解析与画框"""
    import onnxruntime as ort
    sess = ort.InferenceSession(ONNX, providers=["CPUExecutionProvider"])
    x = cv2.resize(img_bgr, (640, 640)).astype(np.float32) / 255.0
    x = x.transpose(2, 0, 1)[None]
    out = sess.run(None, {"images": x})[0][0]        # (84,8400)
    cls = out[4:]
    conf_map = cls.max(0)
    cid = cls.argmax(0)
    keep = conf_map > conf
    boxes = out[:4].T[keep]
    h0, w0 = img_bgr.shape[:2]
    for (cx, cy, w, h) in boxes:
        x1 = int((cx - w / 2) * w0 / 640)
        y1 = int((cy - h / 2) * h0 / 640)
        x2 = int((cx + w / 2) * w0 / 640)
        y2 = int((cy + h / 2) * h0 / 640)
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 0, 255), 2)
    return img_bgr, int(keep.sum())

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-1 解析: cls = out[4:];
>   conf=cls.max(0), cid=cls.argmax(0);
>   keep=conf>阈值
> - （在此处编写你的代码——依据上方注释/文档字符串）
> - TODO-2: 循环调用runner, 统计20张图总检出数, 填入工单表格
> - TODO-3: warmup 3帧后计时20帧, 打印平均FPS(只计推理段!)

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`CPU推理+可视化完成；阈值实验完整`，优秀(B+)=`FPS≥20且检出稳定；权衡分析有理`

### Starter 4：`def pick_backend():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def pick_backend():
    try:
        import ultralytics  # noqa
        return "ultralytics"
    except ImportError:
        return "onnx"

### Starter 5：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def cv_count(img_bgr):
    """【参考答案】传统视觉对照基线: 背景偏离+连通域（预训练权重领域不匹配时的工程解）"""
    f = img_bgr.astype(np.float32)
    bg = np.full(f.shape[:2], 190.0, np.float32)
    bg[40:280, 40:280] = 150.0
    dev = np.max([np.abs(f[:, :, c] - bg) for c in range(3)], axis=0)
    mask = (dev > 15).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    return sum(1 for i in range(1, n) if stats[i][4] >= 25)


def truth_count(path):
    """标注(YOLO归一化txt)中的真值缺陷个数——阈值实验表格的分母"""
    lp = path.replace("images", "labels").replace(".npy", ".txt")
    return sum(1 for ln in open(lp, encoding="utf-8") if ln.strip()) if os.path.exists(lp) else 0


def build_infer(backend):
    """会话级推理函数: 模型只加载一次, 返回检出数(不画框), 支持 conf/imgsz"""
    if backend == "ultralytics":
        from ultralytics import YOLO
        model = YOLO(W_PATH)

        def infer(img_bgr, conf=0.4, imgsz=320):
            r = model.predict(img_bgr, conf=conf, imgsz=imgsz, device="cpu", verbose=False)
            return len(r[0].boxes)
        return infer, "dynamic"
    import onnxruntime as ort
    sess = ort.InferenceSession(ONNX, providers=["CPUExecutionProvider"])
    static = sess.get_inputs()[0].shape
    fixed = isinstance(static[2], int) and static[2] == static[3]

    def infer(img_bgr, conf=0.4, imgsz=320):
        size = static[2] if fixed else imgsz          # 静态图只能按导出尺寸喂
        x = cv2.resize(img_bgr, (size, size)).astype(np.float32) / 255.0
        out = sess.run(None, {"images": x.transpose(2, 0, 1)[None]})[0][0]
        return int((out[4:].max(0) > conf).sum())
    return infer, (str(static[2]) if fixed else "dynamic")


def main():
    backend = pick_backend()
    runner = run_ultralytics if backend == "ultralytics" else run_onnx
    infer, imgsz_mode = build_infer(backend)
    print("后端:", backend, "| 输入尺寸通道:", imgsz_mode)

    # ---- 1) 推理可视化(前5张) ----
    for p in IMGS[:5]:
        img = cv2.cvtColor(np.load(p).astype(np.uint8), cv2.COLOR_RGB2BGR)
        anno, n = runner(img)
        cv2.imwrite("yolo_" + p.split("/")[-1].replace(".npy", ".png"), anno)
        print(p.split("/")[-1], "检出", n)

    # ---- 2) 阈值实验: conf∈{0.25,0.4,0.6,0.8} ---- 【参考答案】TODO-2
    print("\n===== 置信度阈值实验(20张图·YOLO vs 传统视觉 vs 真值) =====")
    n_img = min(20, len(IMGS))
    truth = sum(truth_count(p) for p in IMGS[:n_img])
    print(f"{'conf':<7}{'YOLO检出':>10}{'传统视觉':>10}{'真值':>7}{'YOLO召回':>10}")
    conf_rows = {}
    for conf in (0.25, 0.4, 0.6, 0.8):
        yolo = cv_ = 0
        for p in IMGS[:n_img]:
            img = cv2.cvtColor(np.load(p).astype(np.uint8), cv2.COLOR_RGB2BGR)
            yolo += infer(img, conf=conf)
            cv_ += cv_count(img)
        conf_rows[conf] = (yolo, cv_)
        print(f"{conf:<7}{yolo:>10}{cv_:>10}{truth:>7}{yolo / max(truth, 1):>9.0%}")

    # ---- 3) FPS测试 ---- 【参考答案】TODO-3
    probe = cv2.cvtColor(np.load(IMGS[0]).astype(np.uint8), cv2.COLOR_RGB2BGR)
    sizes = (160, 320, 640) if imgsz_mode == "dynamic" else (int(imgsz_mode),)
    print("\n===== 输入尺寸-帧率实验(warmup 3帧 + 计时20帧·只计推理段) =====")
    fps_rows = {}
    for imgsz in sizes:
        for _ in range(3):
            infer(probe, imgsz=imgsz)
        t0 = time.time()
        for _ in range(20):
            infer(probe, imgsz=imgsz)
        dt = time.time() - t0
        fps_rows[imgsz] = 20 / dt
        print(f"imgsz={imgsz:<6}FPS={20 / dt:>7.1f}  单帧={dt / 20 * 1000:>7.1f}ms")
    for _ in range(3):
        cv_count(probe)
    t0 = time.time()
    for _ in range(20):
        cv_count(probe)
    fps_cv = 20 / (time.time() - t0)
    print(f"传统视觉       FPS={fps_cv:>7.1f}  单帧={(time.time() - t0) / 20 * 1000:>7.1f}ms")
    if imgsz_mode != "dynamic":
        print(f"说明: 离线ONNX为导出时的静态{imgsz_mode}×{imgsz_mode}输入, imgsz三档对比需ultralytics通道"
              f"（环境也是选型维度——机房无torch时按单档实测, 结论口径写入工单）。")

    # ---- 4) 决策矩阵 ---- 【参考答案】
    best_yolo = max(fps_rows.values())
    reach_b = best_yolo >= 20 or fps_cv >= 20
    print(f"\n决策: YOLO最优档FPS={best_yolo:.1f}, 传统视觉FPS={fps_cv:.1f} → "
          f"{'B层FPS≥20 由传统视觉达成' if fps_cv >= 20 and best_yolo < 20 else ('B层FPS≥20 达成' if reach_b else '三档均<20FPS, 需batch推理/TensorRT/量化')}；"
          f"conf=0.4 取 YOLO检出{conf_rows[0.4][0]} / 真值{truth}。")
    print("工程结论①: COCO预训练权重在合成工业图上检出为0 —— 不是bug, 而是『领域数据必须微调』的现场证据。")
    print("工程结论②: 标注有限时传统视觉(阈值+连通域)是可交付基线; 但对低对比/细线缺陷失效。")
    print("路径: 传统视觉先上线拿数据 → 修标注口径 → 微调YOLO 提mAP, 再谈FPS优化。")

### Starter 6：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
if __name__ == "__main__":
    main()

## 四、提交要求与避坑
- 提交: `学号_姓名_M3.2.py` 或本notebook(含输出)
- ⚠ 权重从data/weights离线加载，禁止课堂在线下载
- ⚠ FPS计时只包推理段：warmup 3帧后再计时
- ⚠ ONNX输出是(84,8400)：0-3行xywh，4-83行类别分

---

## 教师参考答案（来源：`教师参考/solutions.py` · `m32()`）

> **发放纪律：本附录勿随学生版下发。**

上方学生版 TODO 与下方参考实现的对应关系：

| 学生版函数 | 说明 |
|---|---|
| `main()` | 见下方 `m32()` 内对应逻辑 |

In [ ]:
def m32():
    """M3.2 完成态：双路线对照 —— ① COCO预训练 YOLOv8n（ONNX）② 传统视觉检测

    本课最重要的工程判断：**预训练模型≠万能**。COCO 预训练权重只在真实自然图像上
    有效；本课数据集是合成工业图（纯色几何缺陷），预训练 YOLO 检出为 0 —— 这不是
    bug，而是"领域数据必须微调"的现场证据。在标注有限时，传统视觉（阈值+连通域）
    是可交付的可靠基线。
    """
    import glob
    import os

    import cv2

    IMGS = sorted(glob.glob("data/yolo_dataset/images/*.npy"))[:12]
    CLASSES = ["dent", "stain", "edge_crack"]

    def load_truth(p):
        lp = p.replace("images", "labels").replace(".npy", ".txt")
        out = []
        if os.path.exists(lp):
            for ln in open(lp, encoding="utf-8"):
                c, cx, cy, w, h = ln.split()
                # YOLO 归一化(中心xywh) → 像素(x,y,w,h)
                out.append((int((float(cx) - float(w) / 2) * 320), int((float(cy) - float(h) / 2) * 320),
                            int(float(w) * 320), int(float(h) * 320), CLASSES[int(c)]))
        return out

    def iou(a, b):
        ix = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
        iy = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
        inter = ix * iy
        un = a[2] * a[3] + b[2] * b[3] - inter
        return inter / un if un else 0.0

    def yolo_pretrained(img):
        """路线①：COCO 预训练 YOLOv8n（ONNX）—— 学生主线"""
        try:
            import onnxruntime as ort
        except ImportError:
            return None
        sess = ort.InferenceSession("data/weights/yolov8n.onnx",
                                    providers=["CPUExecutionProvider"])
        x = cv2.resize(img, (640, 640)).astype(np.float32) / 255.0
        out = sess.run(None, {"images": x.transpose(2, 0, 1)[None]})[0][0]
        return int((out[4:].max(0) > 0.4).sum())

    def cv_baseline(img):
        """路线②：传统视觉（背景建模 + 逐通道偏离 + 连通域）—— 工程解"""
        f = img.astype(np.float32)
        bg = np.full(f.shape[:2], 190.0, np.float32)
        bg[40:280, 40:280] = 150.0                       # 产品区背景
        dev = np.max([np.abs(f[:, :, c] - bg) for c in range(3)], axis=0)
        mask = (dev > 15).astype(np.uint8) * 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
        n, _, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
        return [(int(stats[i][0]), int(stats[i][1]),
                 int(stats[i][2]), int(stats[i][3])) for i in range(1, n) if stats[i][4] >= 25]

    # ---- 逐图统计（真实推理，双路线） ----
    tot_t = tot_p = tot_c = tot_hit = 0
    byc = {c: [0, 0] for c in CLASSES}
    for p in IMGS:
        img = np.load(p)
        truth = load_truth(p)
        ny = yolo_pretrained(img)
        boxes = cv_baseline(img)
        for t in truth:
            byc[t[4]][0] += 1
            if any(iou(t, b) >= 0.3 for b in boxes):
                tot_hit += 1
                byc[t[4]][1] += 1
        tot_t += len(truth)
        tot_c += len(boxes)
        if ny is not None:
            tot_p += ny
    import time
    img0 = np.load(IMGS[0])
    t0 = time.time()
    for _ in range(200):
        cv_baseline(img0)
    fps_cv = 200 / (time.time() - t0)

    print("=== 工单 M3.2 YOLO缺陷检测 · 完成态（双路线） ===")
    print(f"数据集: 合成工业图 {len(IMGS)} 张 · 真值框 {tot_t} 个（三类缺陷）")
    print(f"① COCO预训练YOLOv8n: 检出 {tot_p} 个（预训练权重领域不匹配）")
    print(f"② 传统视觉(阈值+连通域): 检出 {tot_c} / 匹配 {tot_hit}（召回 {tot_hit / max(tot_t, 1):.0%}）≈{fps_cv:.0f}FPS")
    print("分类召回: " + " ".join(f"{c} {v[1]}/{v[0]}" for c, v in byc.items()))
    print("结论①: 预训练模型必须领域微调（迁移学习）")
    print("结论②: 传统视觉仅对高对比缺陷有效，低对比/细线失效")
    print("发现③: edge_crack 真值框约27px vs 实际细线约4px —— 标注口径须与目标尺度一致")
    print("路径④: 传统视觉上线拿数据 → 修标注 → 微调YOLO 提mAP")
    print("要点: 两条路线都不够好，正是本课要学'学习型检测器'的原因。")

    # ---- 明细（供工单/答辩附录，不进PPT效果图） ----
    print("\n[明细] 逐图对照:")
    for p in IMGS[:4]:
        img = np.load(p)
        truth = load_truth(p)
        boxes = cv_baseline(img)
        hits = sum(1 for t in truth if any(iou(t, b) >= 0.3 for b in boxes))
        print(f"  {os.path.basename(p)}: 真值{len(truth)} 传统视觉检出{len(boxes)} 匹配{hits}")

## 运行与验证

上方参考实现可直接运行（合成数据、CPU 离线、结果可复现）。
多解法对比见《多解法手册.md》。